In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02010030rest 20160324 1054..csv
/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02020025rest 20150713 1519..csv
/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02010013rest 20150703 1333..csv
/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02020016rest 20150701 1040..csv
/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02020015_rest 20150630 1527.csv
/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02010022restnew 20150724 14.csv
/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2/02020027rest 20150713 1049..csv
/kaggle/input/datasets/mimi

In [7]:
# Ignore all warnings
import warnings
warnings.filterwarnings('ignore')

# **LOSO_CONFIG**

In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# Input data files are available in the read-only "../input/" directory
# Output data files are available in the read-only "../output/" directory

"""
LOSO-CV Framework v3 for MODMA EEG Classification
===================================================
Shared utility module for Leave-One-Subject-Out Cross-Validation across
all deep learning, hybrid, and traditional ML models on the MODMA
resting-state EEG dataset.

v3 improvements over v2:
  - Spectral feature extraction (Welch PSD, 5 bands x 13 features/channel)
  - Data-driven channel selection (top-25 by Cohen's d, per-fold)
  - Domain-knowledge channel selection (19 standard 10-20 positions)
  - Traditional ML LOSO-CV (SVM, L1-LogReg, RF, XGBoost, ensemble)
  - Multi-modal DL: dual-input (raw EEG + spectral features) architecture
  - Data augmentation: SMOTE (ML), feature-space noise, window bootstrap (DL)
  - Voting threshold lowered to 0.50 (v2's 0.55 overcorrected to HC bias)
  - ML-compatible consolidated plots (skip training curves for sklearn models)

v2 improvements (retained):
  - Subject-wise validation split (1 MDD + 1 HC held out for early stopping)
  - Strong regularization: L2, Gaussian noise, label smoothing, dropout
  - Per-window, per-channel z-score normalization (raw EEG branch only)
  - Class weights (balanced) to counteract 22 MDD vs 13 HC imbalance
  - ReduceLROnPlateau scheduler for adaptive learning rate
  - Subject-level majority voting as primary metric
  - Dual reporting: subject-level (primary) + window-level (secondary)

Usage: Paste this entire file into the first code cell of each Kaggle notebook.
"""

import os
import json
import gc
import warnings
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from scipy.signal import welch
from scipy.stats import bootstrap as scipy_bootstrap
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Input, Dense, Dropout, Concatenate, GaussianNoise, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model

# SMOTE (optional import — only needed for ML augmentation)
try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False

warnings.filterwarnings("ignore")
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'

# Phylo color palette for plots
PHYLO_COLORS = ['#0279EE', '#FF9400', '#75A025', '#FD9BED', '#E9ED4C', '#000000']

# ============================================================================
# REGULARIZATION CONSTANTS (v2)
# ============================================================================
L2_REG = tf.keras.regularizers.l2(1e-4)
GAUSSIAN_NOISE_STD = 0.1
LABEL_SMOOTHING = 0.1
DROPOUT_RATE = 0.5
SPATIAL_DROPOUT = 0.4
VOTING_THRESHOLD = 0.50  # v3: lowered from 0.55 (v2 overcorrected to HC bias)
LEARNING_RATE = 0.0005
ES_PATIENCE = 15

# ============================================================================
# EGI GSN-128 → STANDARD 10-20 CHANNEL MAPPING
# ============================================================================
# Computed by matching MNE-Python GSN-HydroCel-128.sfp 3D coordinates to
# standard 10-20 positions via Euclidean distance. All 19 positions within
# 1-2.5 cm of their GSN equivalents. Source: EGI technical note (Luu & Ferree 2005).
GSN128_TO_1020 = {
    6: 'Fz', 7: 'Cz', 8: 'Fp2', 20: 'F3', 25: 'Fp1',
    33: 'F7', 36: 'C3', 39: 'T7', 52: 'P3', 58: 'P7',
    62: 'Pz', 69: 'O1', 89: 'O2', 92: 'P4', 96: 'P8',
    104: 'C4', 115: 'T8', 118: 'F4', 122: 'F8',
}
# 0-based indices for array slicing (electrode_num - 1)
INDICES_1020 = sorted([k - 1 for k in GSN128_TO_1020.keys()])

# ============================================================================
# SPECTRAL FEATURE CONSTANTS
# ============================================================================
FS = 250  # MODMA sampling rate (Hz)
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 45),
}
BAND_NAMES = list(BANDS.keys())
N_BANDS = len(BANDS)
N_SPECTRAL_FEATURES = N_BANDS * 2 + 3  # 5 abs + 5 rel + 3 ratios = 13
SPECTRAL_FEATURE_NAMES = (
    [f'{b}_abs' for b in BAND_NAMES] +
    [f'{b}_rel' for b in BAND_NAMES] +
    ['theta_beta_ratio', 'alpha_beta_ratio', 'theta_alpha_ratio']
)


# ============================================================================
# 1. DATA PREPARATION
# ============================================================================

def prepare_loso_data(npz_path):
    """
    Load the preprocessed npz and reconstruct the full dataset with subject groups.

    CRITICAL FIX: The original LOSO-CV cell checked for a 'groups' key that doesn't
    exist. This function reconstructs groups from 'subjects_train' and 'subjects_test',
    which ARE saved by the preprocessing cell.

    Returns:
        X: (n_samples, 250, 128) float32
        y: (n_samples,) int32  (1=MDD, 0=HC)
        groups: (n_samples,) array of subject ID strings
        feature_cols: list of channel names
    """
    data = np.load(npz_path, allow_pickle=True)
    X = np.concatenate((data['X_train'], data['X_test']), axis=0)
    y = np.concatenate((data['y_train'], data['y_test']), axis=0)
    groups = np.concatenate((data['subjects_train'], data['subjects_test']), axis=0)
    feature_cols = list(data['feature_columns']) if 'feature_columns' in data else None

    n_subjects = len(np.unique(groups))
    print(f"  Loaded {X.shape[0]} samples, {n_subjects} subjects, {X.shape[2]} channels")
    print(f"  MDD subjects: {len(np.unique(groups[y == 1]))}, HC subjects: {len(np.unique(groups[y == 0]))}")
    return X, y, groups, feature_cols


def prepare_spectral_loso_data(npz_path):
    """
    Load npz and reconstruct the full spectral feature dataset with subject groups.

    Returns:
        X_spec: (n_samples, 128, 13) float32 — spectral features per channel
        y: (n_samples,) int32  (1=MDD, 0=HC)
        groups: (n_samples,) array of subject ID strings
        feature_names: list of 13 spectral feature names
    """
    data = np.load(npz_path, allow_pickle=True)
    X_spec = np.concatenate((data['spectral_train'], data['spectral_test']), axis=0)
    y = np.concatenate((data['y_train'], data['y_test']), axis=0)
    groups = np.concatenate((data['subjects_train'], data['subjects_test']), axis=0)
    feature_names = list(data['spectral_feature_names']) if 'spectral_feature_names' in data else SPECTRAL_FEATURE_NAMES

    n_subjects = len(np.unique(groups))
    print(f"  Loaded spectral: {X_spec.shape[0]} samples, {n_subjects} subjects, {X_spec.shape[1]} channels, {X_spec.shape[2]} features")
    return X_spec, y, groups, feature_names


# ============================================================================
# 2. INPUT SHAPING
# ============================================================================

def shape_input(X, input_mode):
    """Reshape data according to model's expected input format."""
    if input_mode == '3d':
        return X  # (samples, 250, 128)
    elif input_mode == '4d':
        return X[..., np.newaxis]  # (samples, 250, 128, 1)
    elif input_mode == 'flat':
        return X.reshape(X.shape[0], -1)  # (samples, 32000)
    else:
        raise ValueError(f"Unknown input_mode: {input_mode}")


def _zscore_per_window(X_3d):
    """Per-window, per-channel z-score normalization. Each window independently."""
    mean = X_3d.mean(axis=1, keepdims=True)  # (samples, 1, channels)
    std = X_3d.std(axis=1, keepdims=True)
    std = np.where(std == 0, 1.0, std)
    X_norm = (X_3d - mean) / std
    return np.nan_to_num(X_norm, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def preprocess_fold(X_train, X_test, input_mode, X_val=None):
    """
    Per-fold isolated preprocessing:
    1. NaN imputation with training fold means (applied to train, val, test)
    2. Per-window, per-channel z-score normalization (each set independently)
    3. Reshape according to input_mode

    If X_val is provided, returns (X_train_proc, X_val_proc, X_test_proc).
    Otherwise returns (X_train_proc, X_test_proc) for backward compatibility.

    The per-window normalization is critical for cross-subject generalization:
    each EEG window is independently z-scored per channel, removing inter-subject
    amplitude differences while preserving temporal pattern shape. This follows
    the approach of Vaniya et al. (IEEE JBHI 2025) who achieved 90% LOSO accuracy
    with per-segment z-score normalization.
    """
    n_channels = X_train.shape[2]

    # --- NaN imputation (training means applied to all sets) ---
    train_flat = X_train.reshape(-1, n_channels)
    col_means = np.nanmean(train_flat, axis=0)
    nan_idx = np.where(np.isnan(train_flat))
    train_flat[nan_idx] = np.take(col_means, nan_idx[1])

    test_flat = X_test.reshape(-1, n_channels)
    nan_idx_t = np.where(np.isnan(test_flat))
    test_flat[nan_idx_t] = np.take(col_means, nan_idx_t[1])

    X_train_3d = train_flat.reshape(X_train.shape)
    X_test_3d = test_flat.reshape(X_test.shape)

    # --- Per-window z-score normalization (independent per set) ---
    X_train_norm = _zscore_per_window(X_train_3d)
    X_test_norm = _zscore_per_window(X_test_3d)

    if X_val is not None:
        val_flat = X_val.reshape(-1, n_channels)
        nan_idx_v = np.where(np.isnan(val_flat))
        val_flat[nan_idx_v] = np.take(col_means, nan_idx_v[1])
        X_val_3d = val_flat.reshape(X_val.shape)
        X_val_norm = _zscore_per_window(X_val_3d)
        return (shape_input(X_train_norm, input_mode),
                shape_input(X_val_norm, input_mode),
                shape_input(X_test_norm, input_mode))

    return shape_input(X_train_norm, input_mode), shape_input(X_test_norm, input_mode)


# ============================================================================
# 2b. SPECTRAL FEATURE CHANNEL SELECTION & PREPROCESSING
# ============================================================================

def select_top_channels(X_spec_train, y_train, n_channels=25):
    """
    Select top-N channels by mean absolute Cohen's d across all spectral features.
    Computed on TRAINING data only (per-fold) to prevent test-set leakage.

    Args:
        X_spec_train: (n_train_samples, 128, 13) spectral features
        y_train: (n_train_samples,) labels (1=MDD, 0=HC)
        n_channels: number of channels to select
    Returns:
        channel_indices: sorted array of 0-based channel indices
    """
    n_samples, n_ch, n_feat = X_spec_train.shape
    mdd_mask = y_train == 1
    hc_mask = y_train == 0

    mean_d = np.zeros(n_ch)
    for ch in range(n_ch):
        d_per_feat = []
        for f in range(n_feat):
            mdd_vals = X_spec_train[mdd_mask, ch, f]
            hc_vals = X_spec_train[hc_mask, ch, f]
            pooled_std = np.sqrt((np.var(mdd_vals, ddof=1) + np.var(hc_vals, ddof=1)) / 2)
            if pooled_std > 1e-10:
                d = (np.mean(mdd_vals) - np.mean(hc_vals)) / pooled_std
            else:
                d = 0.0
            d_per_feat.append(abs(d))
        mean_d[ch] = np.mean(d_per_feat)

    top_indices = np.argsort(mean_d)[::-1][:n_channels]
    return np.sort(top_indices), mean_d


def preprocess_spectral_fold(X_spec_train, X_spec_test, channel_indices, X_spec_val=None):
    """
    Per-fold spectral feature preprocessing:
    1. Select channels (indices from select_top_channels or INDICES_1020)
    2. Flatten to (n_samples, n_selected * 13)
    3. StandardScaler fit on train, transform on val and test

    Returns (train, test) or (train, val, test).
    """
    X_tr = X_spec_train[:, channel_indices, :].reshape(X_spec_train.shape[0], -1)
    X_te = X_spec_test[:, channel_indices, :].reshape(X_spec_test.shape[0], -1)

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_te = scaler.transform(X_te)

    if X_spec_val is not None:
        X_va = X_spec_val[:, channel_indices, :].reshape(X_spec_val.shape[0], -1)
        X_va = scaler.transform(X_va)
        return X_tr.astype(np.float32), X_va.astype(np.float32), X_te.astype(np.float32)

    return X_tr.astype(np.float32), X_te.astype(np.float32)
# ============================================================================

def mc_dropout_predict(model, X, T=50):
    """
    Run T stochastic forward passes with dropout active (training=True).
    Returns mean probability and predictive variance (epistemic uncertainty).
    """
    X_tensor = tf.convert_to_tensor(X, dtype=tf.float32)
    mc_preds = []
    for _ in range(T):
        pred = model(X_tensor, training=True)
        mc_preds.append(pred.numpy())
    mc_preds = np.array(mc_preds)  # (T, n_samples, 1)
    mean_prob = np.mean(mc_preds, axis=0).ravel()
    variance = np.var(mc_preds, axis=0).ravel()
    return mean_prob, variance


def mc_dropout_predict_multimodal(model, X_eeg, X_spec, T=50):
    """
    MC Dropout inference for multi-modal (dual-input) models.
    Runs T stochastic forward passes with dropout active (training=True).
    """
    X_eeg_t = tf.convert_to_tensor(X_eeg, dtype=tf.float32)
    X_spec_t = tf.convert_to_tensor(X_spec, dtype=tf.float32)
    mc_preds = []
    for _ in range(T):
        pred = model([X_eeg_t, X_spec_t], training=True)
        mc_preds.append(pred.numpy())
    mc_preds = np.array(mc_preds)
    mean_prob = np.mean(mc_preds, axis=0).ravel()
    variance = np.var(mc_preds, axis=0).ravel()
    return mean_prob, variance


# ============================================================================
# 4. CLINICAL METRICS (Reviewer-Required)
# ============================================================================

def wilson_score_ci(p, n, z=1.96):
    """Wilson score interval for a proportion."""
    denom = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denom
    margin = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return center - margin, center + margin


def compute_clinical_metrics(oof_y_true, oof_y_pred, oof_y_pred_prob, per_fold_results,
                             oof_subject_true=None, oof_subject_pred=None,
                             oof_subject_pred_prob=None):
    """
    Compute reviewer-required LOSO-CV clinical metrics at both window and subject level.

    Window-level (secondary):
    - LOSO accuracy (OOF pooled)
    - Standard deviation (per-fold)
    - Sensitivity (MDD recall)
    - Specificity (HC recall)
    - 95% confidence intervals (Wilson + bootstrap)

    Subject-level (primary):
    - Subject-level accuracy (majority voting: mean prob > 0.55 → MDD)
    - Subject-level sensitivity/specificity/F1/ROC AUC
    - 95% CI (Wilson)
    """
    oof_y_true = np.array(oof_y_true)
    oof_y_pred = np.array(oof_y_pred)
    oof_y_pred_prob = np.array(oof_y_pred_prob)

    # --- OOF pooled metrics (window-level) ---
    n_total = len(oof_y_true)
    oof_accuracy = accuracy_score(oof_y_true, oof_y_pred)
    cm = confusion_matrix(oof_y_true, oof_y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # MDD recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # HC recall
    precision_mdd = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1_mdd = 2 * precision_mdd * sensitivity / (precision_mdd + sensitivity) if (precision_mdd + sensitivity) > 0 else 0.0
    f1_hc = f1_score(oof_y_true, oof_y_pred, pos_label=0, zero_division=0)
    f1_macro = f1_score(oof_y_true, oof_y_pred, average='macro', zero_division=0)
    roc_auc = roc_auc_score(oof_y_true, oof_y_pred_prob)

    # --- Wilson score CI on OOF accuracy ---
    wilson_lo, wilson_hi = wilson_score_ci(oof_accuracy, n_total)

    # --- Per-fold accuracy statistics ---
    fold_accs = [r['accuracy'] for r in per_fold_results]
    fold_acc_mean = np.mean(fold_accs)
    fold_acc_sd = np.std(fold_accs, ddof=1)

    # --- Bootstrap CI on per-fold accuracy mean ---
    rng = np.random.default_rng(42)
    n_boot = 10000
    boot_means = []
    for _ in range(n_boot):
        sample = rng.choice(fold_accs, size=len(fold_accs), replace=True)
        boot_means.append(np.mean(sample))
    boot_ci_lo = np.percentile(boot_means, 2.5)
    boot_ci_hi = np.percentile(boot_means, 97.5)

    # --- Per-fold sensitivity/specificity ---
    fold_sens = [r['sensitivity'] for r in per_fold_results if r['sensitivity'] is not None]
    fold_spec = [r['specificity'] for r in per_fold_results if r['specificity'] is not None]

    metrics = {
        'oof_accuracy': float(oof_accuracy),
        'oof_accuracy_95ci_wilson': [float(wilson_lo), float(wilson_hi)],
        'per_fold_accuracy_mean': float(fold_acc_mean),
        'per_fold_accuracy_sd': float(fold_acc_sd),
        'per_fold_accuracy_95ci_bootstrap': [float(boot_ci_lo), float(boot_ci_hi)],
        'sensitivity_mdd': float(sensitivity),
        'specificity_hc': float(specificity),
        'precision_mdd': float(precision_mdd),
        'f1_mdd': float(f1_mdd),
        'f1_hc': float(f1_hc),
        'f1_macro': float(f1_macro),
        'roc_auc_oof': float(roc_auc),
        'n_total_oof_samples': int(n_total),
        'confusion_matrix': [[int(tn), int(fp)], [int(fn), int(tp)]],
        'per_fold_sensitivity_mean': float(np.mean(fold_sens)) if fold_sens else None,
        'per_fold_sensitivity_sd': float(np.std(fold_sens, ddof=1)) if fold_sens else None,
        'per_fold_specificity_mean': float(np.mean(fold_spec)) if fold_spec else None,
        'per_fold_specificity_sd': float(np.std(fold_spec, ddof=1)) if fold_spec else None,
    }

    # --- Subject-level metrics (primary) ---
    if oof_subject_true is not None and oof_subject_pred is not None and oof_subject_pred_prob is not None:
        subj_true = np.array(oof_subject_true)
        subj_pred = np.array(oof_subject_pred)
        subj_prob = np.array(oof_subject_pred_prob)
        n_subjects = len(subj_true)

        subject_accuracy = accuracy_score(subj_true, subj_pred)
        subj_cm = confusion_matrix(subj_true, subj_pred, labels=[0, 1])
        s_tn, s_fp, s_fn, s_tp = subj_cm.ravel()

        subject_sensitivity = s_tp / (s_tp + s_fn) if (s_tp + s_fn) > 0 else 0.0
        subject_specificity = s_tn / (s_tn + s_fp) if (s_tn + s_fp) > 0 else 0.0
        subject_f1_macro = f1_score(subj_true, subj_pred, average='macro', zero_division=0)
        subject_roc_auc = roc_auc_score(subj_true, subj_prob) if len(np.unique(subj_true)) > 1 else 0.5

        subj_wilson_lo, subj_wilson_hi = wilson_score_ci(subject_accuracy, n_subjects)

        metrics['subject_accuracy'] = float(subject_accuracy)
        metrics['subject_sensitivity_mdd'] = float(subject_sensitivity)
        metrics['subject_specificity_hc'] = float(subject_specificity)
        metrics['subject_f1_macro'] = float(subject_f1_macro)
        metrics['subject_roc_auc'] = float(subject_roc_auc)
        metrics['subject_accuracy_95ci_wilson'] = [float(subj_wilson_lo), float(subj_wilson_hi)]
        metrics['subject_confusion_matrix'] = [[int(s_tn), int(s_fp)], [int(s_fn), int(s_tp)]]
        metrics['n_subjects'] = int(n_subjects)
    else:
        # Backward-compatible defaults
        metrics['subject_accuracy'] = None
        metrics['subject_sensitivity_mdd'] = None
        metrics['subject_specificity_hc'] = None
        metrics['subject_f1_macro'] = None
        metrics['subject_roc_auc'] = None
        metrics['subject_accuracy_95ci_wilson'] = None
        metrics['subject_confusion_matrix'] = None
        metrics['n_subjects'] = None

    return metrics


def compute_per_fold_metrics(y_true, y_pred, y_pred_prob, subject_id, subject_label):
    """Compute per-fold window-level and subject-level metrics for a single held-out subject."""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_pred_prob = np.array(y_pred_prob)
    acc = accuracy_score(y_true, y_pred)

    # Per-fold sensitivity/specificity (window-level)
    # If subject is MDD (label=1), all samples are class 1 → specificity undefined
    # If subject is HC (label=0), all samples are class 0 → sensitivity undefined
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else None
    specificity = tn / (tn + fp) if (tn + fp) > 0 else None

    # Subject-level majority voting (mean probability > VOTING_THRESHOLD → MDD)
    subject_pred_prob = float(np.mean(y_pred_prob))
    subject_pred = 1 if subject_pred_prob > VOTING_THRESHOLD else 0
    subject_accuracy = 1.0 if subject_pred == subject_label else 0.0

    return {
        'subject': str(subject_id),
        'label': int(subject_label),
        'accuracy': float(acc),
        'sensitivity': float(sensitivity) if sensitivity is not None else None,
        'specificity': float(specificity) if specificity is not None else None,
        'n_samples': int(len(y_true)),
        'n_correct': int(np.sum(y_true == y_pred)),
        'subject_pred_prob': subject_pred_prob,
        'subject_pred': int(subject_pred),
        'subject_accuracy': float(subject_accuracy),
    }


# ============================================================================
# 4b. MULTI-MODAL WRAPPER & DATA AUGMENTATION
# ============================================================================

def wrap_multimodal(build_fn, spectral_dim, spectral_units=64):
    """
    Wrap a single-input DL model builder to create a dual-input model.
    EEG branch: original model up to penultimate layer (before final Dense(1,sigmoid)).
    Spectral branch: GaussianNoise -> Dense(spectral_units) -> Dropout.
    Concatenate -> Dense(spectral_units) -> Dropout -> Dense(1, sigmoid).

    For special='tcn_knn' where build_fn returns (model, feature_extractor):
    returns (wrapped_model, feature_extractor_mm) where feature_extractor_mm
    outputs the concatenated EEG+spectral features for KNN.
    """
    def build_multimodal(input_shape):
        result = build_fn(input_shape)
        # Handle tcn_knn case where build_fn returns (model, feature_extractor)
        if isinstance(result, tuple):
            original, _ = result
        else:
            original = result

        # Extract penultimate layer output (layer before final Dense(1, sigmoid))
        eeg_features = original.layers[-2].output

        # Spectral branch
        spectral_input = Input(shape=(spectral_dim,), name='spectral_input')
        s = GaussianNoise(GAUSSIAN_NOISE_STD)(spectral_input)
        s = Dense(spectral_units, activation='relu', kernel_regularizer=L2_REG)(s)
        s = Dropout(DROPOUT_RATE)(s)

        # Concatenate and classify
        combined = Concatenate()([eeg_features, s])
        combined = Dense(spectral_units, activation='relu', kernel_regularizer=L2_REG)(combined)
        combined = Dropout(DROPOUT_RATE)(combined)
        outputs = Dense(1, activation='sigmoid')(combined)

        model = Model(original.inputs + [spectral_input], outputs)
        # Reuse original optimizer config to preserve model-specific settings
        # (e.g., GAN's beta_1=0.5, LSTM's clipnorm=1.0)
        try:
            opt_config = original.optimizer.get_config()
            new_optimizer = type(original.optimizer)(**opt_config)
        except Exception:
            new_optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)
        model.compile(
            optimizer=new_optimizer,
            loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
            metrics=['accuracy'])

        # For tcn_knn: return (model, feature_extractor_mm)
        if isinstance(result, tuple):
            feature_extractor_mm = Model(model.inputs, combined)
            return model, feature_extractor_mm

        return model
    return build_multimodal


class SubjectBootstrapGenerator(tf.keras.utils.Sequence):
    """
    Custom data generator that bootstrap-resamples windows within each
    training subject per epoch. Ensures balanced subject representation
    and reduces dominance of subjects with more windows.

    Yields ((X_eeg_batch, X_spec_batch), y_batch) for multi-modal models,
    or (X_eeg_batch, y_batch) for single-input models.

    NOTE: the multi-input side MUST be a tuple, not a list. When Keras
    converts a Sequence into a tf.data pipeline (GeneratorDataAdapter), its
    output_signature inference recurses into tuple/dict structures but
    treats a bare list as an opaque leaf. A list here produces a raw
    `list` object inside output_signature instead of a nested tf.TypeSpec,
    which tf.data.Dataset.from_generator rejects with:
    "TypeError: `output_signature` must contain objects that are subclass
    of `tf.TypeSpec` but found <class 'list'>". This does NOT affect
    model.fit(x=[arr1, arr2], ...) with raw numpy arrays, which goes
    through a different adapter (array-based) that handles lists fine —
    it only bites generator/Sequence-based inputs.
    """
    def __init__(self, X_eeg, y, groups, batch_size, X_spec=None, seed=42):
        self.X_eeg = X_eeg
        self.X_spec = X_spec
        self.y = y
        self.groups = groups
        self.batch_size = batch_size
        self.rng = np.random.default_rng(seed)
        self.unique_subjects = np.unique(groups)
        self._resample()

    def _resample(self):
        """Bootstrap-resample windows within each subject."""
        indices = []
        for subj in self.unique_subjects:
            subj_idx = np.where(self.groups == subj)[0]
            n = len(subj_idx)
            resampled = self.rng.choice(subj_idx, size=n, replace=True)
            indices.extend(resampled.tolist())
        self.indices = np.array(indices)
        self.rng.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_eeg_batch = self.X_eeg[batch_idx]
        y_batch = self.y[batch_idx]
        if self.X_spec is not None:
            X_spec_batch = self.X_spec[batch_idx]
            return (X_eeg_batch, X_spec_batch), y_batch
        return X_eeg_batch, y_batch

    def on_epoch_end(self):
        self._resample()


# ============================================================================
# 5. LOSO-CV RUNNER
# ============================================================================

def run_loso_cv(model_name, build_fn, X, y, groups, input_mode,
                max_epochs, batch_size, output_dir, special=None,
                results_registry=None, X_spectral=None, use_bootstrap=False):
    """
    Run Leave-One-Subject-Out CV for a single model.

    Args:
        model_name: str identifier (e.g., 'TCN', 'TCN_LSTM')
        build_fn: callable(input_shape) -> compiled tf.keras.Model
                   For special='tcn_knn': returns (full_model, feature_extractor)
        X, y, groups: full dataset from prepare_loso_data()
        input_mode: '3d', '4d', or 'flat'
        max_epochs: int, maximum training epochs (early stopping may reduce this)
        batch_size: int
        output_dir: str, base directory for LOSO-CV outputs
        special: None, 'gan', 'mc_dropout', or 'tcn_knn'
        results_registry: list to append results dict (for cross-model plots)
        X_spectral: (n_samples, 128, 13) spectral features for multi-modal DL.
                    If provided, uses 19 10-20 channels (fixed) as second input.
        use_bootstrap: if True, use SubjectBootstrapGenerator for training

    Returns:
        results dict with metrics, histories, per_fold_results
    """
    model_dir = os.path.join(output_dir, model_name)
    os.makedirs(model_dir, exist_ok=True)

    logo = LeaveOneGroupOut()
    unique_subjects = np.unique(groups)
    n_folds = len(unique_subjects)

    fold_histories = {}
    per_fold_results = []
    oof_y_true = []
    oof_y_pred_prob = []
    oof_y_pred = []
    oof_subject_true = []
    oof_subject_pred = []
    oof_subject_pred_prob = []
    epochs_trained_list = []

    print(f"\n{'='*70}")
    print(f"LOSO-CV: {model_name} | {n_folds} folds | input={input_mode} | special={special}")
    print(f"{'='*70}")

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        subject_id = groups[test_idx][0]
        subject_label = int(y[test_idx][0])

        print(f"\n--- Fold {fold+1}/{n_folds} | Subject: {subject_id} | Label: {'MDD' if subject_label==1 else 'HC'} | Samples: {len(test_idx)} ---")

        # --- Prepare fold data ---
        X_test_fold = X[test_idx]
        y_test_fold = y[test_idx]

        # --- Subject-wise validation split (v2) ---
        # Hold out 1 MDD + 1 HC subject from training for early stopping,
        # rotated by fold index so all subjects serve as validation across folds.
        train_subjects = np.unique(groups[train_idx])
        train_subject_labels = np.array([y[groups == s][0] for s in train_subjects])
        train_mdd = train_subjects[train_subject_labels == 1]
        train_hc = train_subjects[train_subject_labels == 0]

        val_mdd = train_mdd[fold % len(train_mdd)]
        val_hc = train_hc[fold % len(train_hc)]
        val_subjects = np.array([val_mdd, val_hc])

        val_mask = np.isin(groups[train_idx], val_subjects)
        inner_train_idx = train_idx[~val_mask]
        val_idx = train_idx[val_mask]

        X_train_inner = X[inner_train_idx]
        X_val_fold = X[val_idx]
        y_train_inner = y[inner_train_idx]
        y_val_fold = y[val_idx]

        print(f"  Val subjects: {val_mdd} (MDD), {val_hc} (HC) | Inner train: {len(inner_train_idx)} | Val: {len(val_idx)}")

        # Preprocess (isolated per fold, with validation set)
        X_train_proc, X_val_proc, X_test_proc = preprocess_fold(
            X_train_inner, X_test_fold, input_mode, X_val=X_val_fold)
        input_shape = X_train_proc.shape[1:]

        # --- Spectral feature preprocessing (multi-modal) ---
        is_multimodal = X_spectral is not None
        # if is_multimodal:
        #     X_spec_train = preprocess_spectral_fold(
        #         X_spectral[inner_train_idx], X_spectral[test_idx],
        #         np.array(INDICES_1020), X_val=X_spectral[val_idx])
        #     X_spec_train_proc, X_spec_val_proc, X_spec_test_proc = X_spec_train
        # if is_multimodal:
        #     X_spec_train, X_spec_val, X_spec_test = preprocess_spectral_fold(
        #         X_spectral[inner_train_idx],
        #         X_spectral[test_idx],
        #         np.array(INDICES_1020),
        #     X_val=X_spectral[val_idx])
        #     X_spec_train_proc, X_spec_val_proc, X_spec_test_proc = X_spec_train

        if is_multimodal:
            X_spec_train_proc, X_spec_val_proc, X_spec_test_proc = preprocess_spectral_fold(
                X_spectral[inner_train_idx],
                X_spectral[test_idx],
            np.array(INDICES_1020),
            X_spec_val=X_spectral[val_idx]
        )
            # X_spec_train_proc, X_spec_val_proc, X_spec_test_proc = X_spec_train
        # --- Checkpoint path ---
        weights_path = os.path.join(model_dir, f'fold_{fold+1}_subject_{subject_id}_best.weights.h5')

        # --- Resumability: skip training if weights exist ---
        skip_training = os.path.exists(weights_path)

        # --- Class weights (balanced) on inner training set (excluding val subjects) ---
        cw = compute_class_weight('balanced', classes=np.unique(y_train_inner), y=y_train_inner)
        class_weight_dict = dict(zip(np.unique(y_train_inner), cw))

        if special == 'tcn_knn':
            # --- TCN+KNN special case ---
            if skip_training:
                print(f"  [RESUME] Weights found, skipping training.")
                full_model, feature_extractor = build_fn(input_shape)
                full_model.load_weights(weights_path)
                epochs_trained = 0
            else:
                full_model, feature_extractor = build_fn(input_shape)

                es = EarlyStopping(monitor='val_loss', patience=ES_PATIENCE, restore_best_weights=True, verbose=1)
                ckpt = ModelCheckpoint(filepath=weights_path, save_weights_only=True,
                                       save_best_only=True, monitor='val_loss', mode='min', verbose=1)
                reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

                if is_multimodal:
                    history = full_model.fit([X_train_proc, X_spec_train_proc], y_train_inner,
                                             epochs=max_epochs, batch_size=batch_size,
                                             validation_data=([X_val_proc, X_spec_val_proc], y_val_fold),
                                             class_weight=class_weight_dict,
                                             callbacks=[es, ckpt, reduce_lr], verbose=1)
                else:
                    history = full_model.fit(X_train_proc, y_train_inner, epochs=max_epochs,
                                             batch_size=batch_size, validation_data=(X_val_proc, y_val_fold),
                                             class_weight=class_weight_dict,
                                             callbacks=[es, ckpt, reduce_lr], verbose=1)
                fold_histories[f'fold_{fold+1}'] = history.history
                epochs_trained = len(history.history['loss'])
                full_model.load_weights(weights_path)

            # Extract features
            if is_multimodal:
                train_features = feature_extractor.predict([X_train_proc, X_spec_train_proc], verbose=0)
                test_features = feature_extractor.predict([X_test_proc, X_spec_test_proc], verbose=0)
            else:
                train_features = feature_extractor.predict(X_train_proc, verbose=0)
                test_features = feature_extractor.predict(X_test_proc, verbose=0)

            # Scale features
            feat_scaler = StandardScaler()
            train_features = feat_scaler.fit_transform(train_features)
            test_features = feat_scaler.transform(test_features)

            # KNN
            knn = KNeighborsClassifier(n_neighbors=5)
            knn.fit(train_features, y_train_inner)
            y_pred_prob = knn.predict_proba(test_features)[:, 1]
            y_pred = (y_pred_prob > 0.5).astype(int)

        elif special == 'mc_dropout':
            # --- MC Dropout BNN special case ---
            if skip_training:
                print(f"  [RESUME] Weights found, skipping training.")
                model = build_fn(input_shape)
                model.load_weights(weights_path)
                epochs_trained = 0
            else:
                model = build_fn(input_shape)

                es = EarlyStopping(monitor='val_loss', patience=ES_PATIENCE, restore_best_weights=True, verbose=1)
                ckpt = ModelCheckpoint(filepath=weights_path, save_weights_only=True,
                                       save_best_only=True, monitor='val_loss', mode='min', verbose=1)
                reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

                if is_multimodal:
                    history = model.fit([X_train_proc, X_spec_train_proc], y_train_inner,
                                        epochs=max_epochs, batch_size=batch_size,
                                        validation_data=([X_val_proc, X_spec_val_proc], y_val_fold),
                                        class_weight=class_weight_dict,
                                        callbacks=[es, ckpt, reduce_lr], verbose=1)
                else:
                    history = model.fit(X_train_proc, y_train_inner, epochs=max_epochs,
                                        batch_size=batch_size, validation_data=(X_val_proc, y_val_fold),
                                        class_weight=class_weight_dict,
                                        callbacks=[es, ckpt, reduce_lr], verbose=1)
                fold_histories[f'fold_{fold+1}'] = history.history
                epochs_trained = len(history.history['loss'])
                model.load_weights(weights_path)

            # MC Dropout inference
            if is_multimodal:
                y_pred_prob, uncertainty = mc_dropout_predict_multimodal(
                    model, X_test_proc, X_spec_test_proc, T=50)
            else:
                y_pred_prob, uncertainty = mc_dropout_predict(model, X_test_proc, T=50)
            y_pred = (y_pred_prob > 0.5).astype(int)

        else:
            # --- Standard model (including 'gan' discriminator-only) ---
            if skip_training:
                print(f"  [RESUME] Weights found, skipping training.")
                model = build_fn(input_shape)
                model.load_weights(weights_path)
                epochs_trained = 0
            else:
                model = build_fn(input_shape)

                es = EarlyStopping(monitor='val_loss', patience=ES_PATIENCE, restore_best_weights=True, verbose=1)
                ckpt = ModelCheckpoint(filepath=weights_path, save_weights_only=True,
                                       save_best_only=True, monitor='val_loss', mode='min', verbose=1)
                reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

                if is_multimodal:
                    if use_bootstrap:
                        gen = SubjectBootstrapGenerator(
                            X_train_proc, y_train_inner, groups[inner_train_idx],
                            batch_size, X_spec=X_spec_train_proc)
                        val_data = ((X_val_proc, X_spec_val_proc), y_val_fold)
                        history = model.fit(gen, epochs=max_epochs, validation_data=val_data,
                                            class_weight=class_weight_dict,
                                            callbacks=[es, ckpt, reduce_lr], verbose=1)
                    else:
                        history = model.fit([X_train_proc, X_spec_train_proc], y_train_inner,
                                            epochs=max_epochs, batch_size=batch_size,
                                            validation_data=([X_val_proc, X_spec_val_proc], y_val_fold),
                                            class_weight=class_weight_dict,
                                            callbacks=[es, ckpt, reduce_lr], verbose=1)
                else:
                    if use_bootstrap:
                        gen = SubjectBootstrapGenerator(
                            X_train_proc, y_train_inner, groups[inner_train_idx],
                            batch_size, X_spec=None)
                        history = model.fit(gen, epochs=max_epochs,
                                            validation_data=(X_val_proc, y_val_fold),
                                            class_weight=class_weight_dict,
                                            callbacks=[es, ckpt, reduce_lr], verbose=1)
                    else:
                        history = model.fit(X_train_proc, y_train_inner, epochs=max_epochs,
                                            batch_size=batch_size, validation_data=(X_val_proc, y_val_fold),
                                            class_weight=class_weight_dict,
                                            callbacks=[es, ckpt, reduce_lr], verbose=1)
                fold_histories[f'fold_{fold+1}'] = history.history
                epochs_trained = len(history.history['loss'])
                model.load_weights(weights_path)

            # Standard inference
            if is_multimodal:
                y_pred_prob = model.predict([X_test_proc, X_spec_test_proc], verbose=0).ravel()
            else:
                y_pred_prob = model.predict(X_test_proc, verbose=0).ravel()
            y_pred = (y_pred_prob > 0.5).astype(int)

        # --- Collect OOF predictions (window-level) ---
        oof_y_true.extend(y_test_fold)
        oof_y_pred_prob.extend(y_pred_prob)
        oof_y_pred.extend(y_pred)
        epochs_trained_list.append(epochs_trained)

        # --- Subject-level majority voting ---
        subject_pred_prob_mean = float(np.mean(y_pred_prob))
        subject_pred = 1 if subject_pred_prob_mean > VOTING_THRESHOLD else 0
        oof_subject_true.append(subject_label)
        oof_subject_pred.append(subject_pred)
        oof_subject_pred_prob.append(subject_pred_prob_mean)

        # --- Per-fold metrics ---
        fold_metrics = compute_per_fold_metrics(y_test_fold, y_pred, y_pred_prob, subject_id, subject_label)
        fold_metrics['fold'] = fold + 1
        fold_metrics['epochs_trained'] = epochs_trained
        per_fold_results.append(fold_metrics)

        print(f"  Window acc: {fold_metrics['accuracy']:.4f} | Subject pred: {'MDD' if subject_pred==1 else 'HC'} (true: {'MDD' if subject_label==1 else 'HC'}) | Epochs: {epochs_trained}")

        # Cleanup
        try:
            del model
        except NameError:
            pass
        try:
            del full_model, feature_extractor
        except NameError:
            pass
        gc.collect()
        tf.keras.backend.clear_session()

    # --- Save fold histories ---
    hist_path = os.path.join(model_dir, f'{model_name}_loso_histories.npz')
    np.savez_compressed(hist_path, **fold_histories)
    print(f"\n  Saved histories: {hist_path}")

    # --- Compute clinical metrics (window-level + subject-level) ---
    metrics = compute_clinical_metrics(
        oof_y_true, oof_y_pred, oof_y_pred_prob, per_fold_results,
        oof_subject_true, oof_subject_pred, oof_subject_pred_prob
    )
    metrics['model_name'] = model_name
    metrics['n_folds'] = n_folds
    metrics['mean_epochs_trained'] = float(np.mean(epochs_trained_list))
    metrics['per_fold_results'] = per_fold_results

    # --- Save metrics ---
    metrics_path = os.path.join(model_dir, f'{model_name}_loso_metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"  Saved metrics: {metrics_path}")

    # --- Print summary ---
    print(f"\n  {'='*60}")
    print(f"  {model_name} LOSO-CV RESULTS:")
    print(f"  {'='*60}")
    print(f"  --- Subject-Level (Primary) ---")
    print(f"  Subject Accuracy:  {metrics['subject_accuracy']:.4f}")
    print(f"  Subject Sens (MDD):{metrics['subject_sensitivity_mdd']:.4f}")
    print(f"  Subject Spec (HC): {metrics['subject_specificity_hc']:.4f}")
    print(f"  Subject F1 Macro:  {metrics['subject_f1_macro']:.4f}")
    print(f"  Subject ROC AUC:   {metrics['subject_roc_auc']:.4f}")
    print(f"  95% CI (Wilson):   [{metrics['subject_accuracy_95ci_wilson'][0]:.4f}, {metrics['subject_accuracy_95ci_wilson'][1]:.4f}]")
    print(f"  --- Window-Level (Secondary) ---")
    print(f"  OOF Accuracy:      {metrics['oof_accuracy']:.4f}")
    print(f"  Per-Fold Acc:      {metrics['per_fold_accuracy_mean']:.4f} ± {metrics['per_fold_accuracy_sd']:.4f}")
    print(f"  95% CI (Wilson):   [{metrics['oof_accuracy_95ci_wilson'][0]:.4f}, {metrics['oof_accuracy_95ci_wilson'][1]:.4f}]")
    print(f"  95% CI (Boot):     [{metrics['per_fold_accuracy_95ci_bootstrap'][0]:.4f}, {metrics['per_fold_accuracy_95ci_bootstrap'][1]:.4f}]")
    print(f"  Sensitivity (MDD): {metrics['sensitivity_mdd']:.4f}")
    print(f"  Specificity (HC):  {metrics['specificity_hc']:.4f}")
    print(f"  F1 Macro:          {metrics['f1_macro']:.4f}")
    print(f"  ROC AUC (OOF):     {metrics['roc_auc_oof']:.4f}")
    print(f"  --- Training ---")
    print(f"  Mean Epochs:       {metrics['mean_epochs_trained']:.1f}")
    print(f"  {'='*60}")

    # --- Build results dict for registry ---
    result = {
        'model_name': model_name,
        'metrics': metrics,
        'histories': fold_histories,
        'oof_y_true': np.array(oof_y_true),
        'oof_y_pred': np.array(oof_y_pred),
        'oof_y_pred_prob': np.array(oof_y_pred_prob),
        'oof_subject_true': np.array(oof_subject_true),
        'oof_subject_pred': np.array(oof_subject_pred),
        'oof_subject_pred_prob': np.array(oof_subject_pred_prob),
        'per_fold_results': per_fold_results,
    }

    if results_registry is not None:
        results_registry.append(result)

    return result


# ============================================================================
# 5b. ML LOSO-CV RUNNER (Spectral Features)
# ============================================================================

def run_spectral_loso_cv(model_name, build_fn, X_spec, y, groups,
                         channel_mode='top25', output_dir=None,
                         results_registry=None, augment=None,
                         voting_threshold=0.50):
    """
    LOSO-CV for traditional ML models using spectral features.

    Args:
        model_name: str identifier (e.g., 'SVM_RBF', 'L1_LogReg')
        build_fn: callable() -> sklearn estimator (fresh instance each call)
        X_spec: (n_samples, 128, 13) spectral features
        y: (n_samples,) labels (1=MDD, 0=HC)
        groups: (n_samples,) subject IDs
        channel_mode: 'top25', '1020', or 'all'
        output_dir: str, base directory for outputs (None = no file saves)
        results_registry: list to append results dict
        augment: None, 'smote', 'noise', or 'smote_noise'
        voting_threshold: float, subject-level voting threshold

    Returns:
        results dict compatible with generate_consolidated_plots / generate_summary_table
    """
    if output_dir:
        model_dir = os.path.join(output_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)

    logo = LeaveOneGroupOut()
    unique_subjects = np.unique(groups)
    n_folds = len(unique_subjects)

    per_fold_results = []
    oof_y_true = []
    oof_y_pred_prob = []
    oof_y_pred = []
    oof_subject_true = []
    oof_subject_pred = []
    oof_subject_pred_prob = []

    aug_label = augment or 'none'
    print(f"\n{'='*70}")
    print(f"ML LOSO-CV: {model_name} | {n_folds} folds | channels={channel_mode} | augment={aug_label}")
    print(f"{'='*70}")

    for fold, (train_idx, test_idx) in enumerate(logo.split(X_spec, y, groups)):
        subject_id = groups[test_idx][0]
        subject_label = int(y[test_idx][0])

        print(f"\n--- Fold {fold+1}/{n_folds} | Subject: {subject_id} | Label: {'MDD' if subject_label==1 else 'HC'} | Samples: {len(test_idx)} ---")

        # --- Channel selection (on training data only) ---
        if channel_mode == 'top25':
            channel_indices, _ = select_top_channels(X_spec[train_idx], y[train_idx], n_channels=25)
        elif channel_mode == '1020':
            channel_indices = np.array(INDICES_1020)
        elif channel_mode == 'all':
            channel_indices = np.arange(X_spec.shape[1])
        else:
            raise ValueError(f"Unknown channel_mode: {channel_mode}")

        # --- Flatten + StandardScaler ---
        X_train_flat, X_test_flat = preprocess_spectral_fold(
            X_spec[train_idx], X_spec[test_idx], channel_indices)
        y_train = y[train_idx]
        y_test = y[test_idx]

        # --- Data augmentation ---
        if augment in ('smote', 'smote_noise') and SMOTE_AVAILABLE:
            smote = SMOTE(random_state=42)
            X_train_flat, y_train = smote.fit_resample(X_train_flat, y_train)
        if augment in ('noise', 'smote_noise'):
            X_train_flat = X_train_flat + np.random.normal(0, 0.01, X_train_flat.shape).astype(np.float32)

        # --- Train ---
        model = build_fn()
        model.fit(X_train_flat, y_train)

        # --- Predict ---
        if hasattr(model, 'predict_proba'):
            y_pred_prob = model.predict_proba(X_test_flat)[:, 1]
        else:
            y_pred_prob = model.decision_function(X_test_flat)
            from sklearn.preprocessing import MinMaxScaler
            y_pred_prob = (y_pred_prob - y_pred_prob.min()) / (y_pred_prob.max() - y_pred_prob.min() + 1e-10)
        y_pred = (y_pred_prob > 0.5).astype(int)

        # --- Collect OOF ---
        oof_y_true.extend(y_test)
        oof_y_pred_prob.extend(y_pred_prob)
        oof_y_pred.extend(y_pred)

        # --- Subject-level voting ---
        subject_pred_prob_mean = float(np.mean(y_pred_prob))
        subject_pred = 1 if subject_pred_prob_mean > voting_threshold else 0
        oof_subject_true.append(subject_label)
        oof_subject_pred.append(subject_pred)
        oof_subject_pred_prob.append(subject_pred_prob_mean)

        # --- Per-fold metrics ---
        fold_metrics = compute_per_fold_metrics(y_test, y_pred, y_pred_prob, subject_id, subject_label)
        fold_metrics['fold'] = fold + 1
        fold_metrics['epochs_trained'] = 0
        fold_metrics['channel_mode'] = channel_mode
        fold_metrics['augment'] = aug_label
        fold_metrics['n_selected_channels'] = len(channel_indices)
        per_fold_results.append(fold_metrics)

        print(f"  Window acc: {fold_metrics['accuracy']:.4f} | Subject pred: {'MDD' if subject_pred==1 else 'HC'} (true: {'MDD' if subject_label==1 else 'HC'})")

    # --- Compute clinical metrics ---
    metrics = compute_clinical_metrics(
        oof_y_true, oof_y_pred, oof_y_pred_prob, per_fold_results,
        oof_subject_true, oof_subject_pred, oof_subject_pred_prob)
    metrics['model_name'] = model_name
    metrics['n_folds'] = n_folds
    metrics['mean_epochs_trained'] = 0.0
    metrics['per_fold_results'] = per_fold_results
    metrics['channel_mode'] = channel_mode
    metrics['augment'] = aug_label

    # --- Save metrics ---
    if output_dir:
        metrics_path = os.path.join(model_dir, f'{model_name}_loso_metrics.json')
        with open(metrics_path, 'w') as f:
            json.dump(metrics, f, indent=2)
        print(f"  Saved metrics: {metrics_path}")

    # --- Print summary ---
    print(f"\n  {'='*60}")
    print(f"  {model_name} ({channel_mode}, {aug_label}) LOSO-CV RESULTS:")
    print(f"  {'='*60}")
    print(f"  --- Subject-Level (Primary) ---")
    print(f"  Subject Accuracy:  {metrics['subject_accuracy']:.4f}")
    print(f"  Subject Sens (MDD):{metrics['subject_sensitivity_mdd']:.4f}")
    print(f"  Subject Spec (HC): {metrics['subject_specificity_hc']:.4f}")
    print(f"  Subject F1 Macro:  {metrics['subject_f1_macro']:.4f}")
    print(f"  Subject ROC AUC:   {metrics['subject_roc_auc']:.4f}")
    print(f"  --- Window-Level (Secondary) ---")
    print(f"  OOF Accuracy:      {metrics['oof_accuracy']:.4f}")
    print(f"  Sensitivity (MDD): {metrics['sensitivity_mdd']:.4f}")
    print(f"  Specificity (HC):  {metrics['specificity_hc']:.4f}")
    print(f"  ROC AUC (OOF):     {metrics['roc_auc_oof']:.4f}")
    print(f"  {'='*60}")

    # --- Build results dict ---
    result = {
        'model_name': model_name,
        'metrics': metrics,
        'histories': {},  # ML models have no training histories
        'oof_y_true': np.array(oof_y_true),
        'oof_y_pred': np.array(oof_y_pred),
        'oof_y_pred_prob': np.array(oof_y_pred_prob),
        'oof_subject_true': np.array(oof_subject_true),
        'oof_subject_pred': np.array(oof_subject_pred),
        'oof_subject_pred_prob': np.array(oof_subject_pred_prob),
        'per_fold_results': per_fold_results,
        'is_ml': True,  # flag for plot compatibility
    }

    if results_registry is not None:
        results_registry.append(result)

    return result


# ============================================================================
# 6. CONSOLIDATED PLOTS
# ============================================================================

def generate_consolidated_plots(results_registry, output_dir, notebook_type='DL'):
    """
    Generate 4 consolidated images showing ALL models together:
    1. Classification reports (subject-level, one image)
    2. ROC curves (subject-level, one image, overlaid)
    3. Train/val accuracy curves (one image, grid of subplots)
    4. Train loss curves (one image, grid of subplots)
    """
    n_models = len(results_registry)
    if n_models == 0:
        print("No results to plot.")
        return

    # --- 1. Classification Reports in One Image (subject-level) ---
    fig, axes = plt.subplots(n_models, 1, figsize=(14, 4 * n_models), dpi=150)
    if n_models == 1:
        axes = [axes]

    for i, result in enumerate(results_registry):
        name = result['model_name']
        # Use subject-level predictions if available, else fall back to window-level
        if 'oof_subject_true' in result and result['oof_subject_true'] is not None:
            y_true = result['oof_subject_true']
            y_pred = result['oof_subject_pred']
        else:
            y_true = result['oof_y_true']
            y_pred = result['oof_y_pred']
        report = classification_report(y_true, y_pred, target_names=['HC (0)', 'MDD (1)'], output_dict=True, zero_division=0)

        # Build table data
        rows = []
        cells = []
        for cls in ['HC (0)', 'MDD (1)']:
            r = report[cls]
            cells.append([cls, f"{r['precision']:.3f}", f"{r['recall']:.3f}", f"{r['f1-score']:.3f}", f"{int(r['support'])}"])
        # Add macro avg
        ma = report['macro avg']
        cells.append(['macro avg', f"{ma['precision']:.3f}", f"{ma['recall']:.3f}", f"{ma['f1-score']:.3f}", f"{int(ma['support'])}"])

        m = result['metrics']
        ax = axes[i]
        ax.axis('off')
        subj_acc = m.get('subject_accuracy')
        subj_sens = m.get('subject_sensitivity_mdd')
        subj_spec = m.get('subject_specificity_hc')
        subj_auc = m.get('subject_roc_auc')
        title_parts = [f"{name}"]
        if subj_acc is not None:
            title_parts.append(f"Subj Acc={subj_acc:.3f}")
            title_parts.append(f"Sens={subj_sens:.3f}")
            title_parts.append(f"Spec={subj_spec:.3f}")
            title_parts.append(f"AUC={subj_auc:.3f}")
        else:
            title_parts.append(f"Win Acc={m['oof_accuracy']:.3f}")
            title_parts.append(f"Sens={m['sensitivity_mdd']:.3f}")
            title_parts.append(f"Spec={m['specificity_hc']:.3f}")
            title_parts.append(f"AUC={m['roc_auc_oof']:.3f}")
        ax.set_title(" | ".join(title_parts), fontsize=11, fontweight='bold', pad=8)
        table = ax.table(cellText=cells, colLabels=['Class', 'Precision', 'Recall', 'F1', 'Support'],
                         loc='center', cellLoc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        table.scale(1, 1.4)
        for (row, col), cell in table.get_celld().items():
            if row == 0:
                cell.set_text_props(fontweight='bold', color='white')
                cell.set_facecolor('#0279EE')
            else:
                cell.set_facecolor('#F2F2F2' if row % 2 == 0 else 'white')

    plt.tight_layout(pad=2.0)
    path = os.path.join(output_dir, f'all_models_classification_reports_{notebook_type}.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {path}")

    # --- 2. ROC Curves in One Image (subject-level) ---
    plt.figure(figsize=(10, 8), dpi=200)
    colors = PHYLO_COLORS * (n_models // len(PHYLO_COLORS) + 1)

    for i, result in enumerate(results_registry):
        name = result['model_name']
        # Use subject-level predictions if available, else fall back to window-level
        if 'oof_subject_true' in result and result['oof_subject_true'] is not None:
            y_true = result['oof_subject_true']
            y_prob = result['oof_subject_pred_prob']
        else:
            y_true = result['oof_y_true']
            y_prob = result['oof_y_pred_prob']
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc = roc_auc_score(y_true, y_prob)
        plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{name} (AUC={auc:.3f})')

    plt.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title(f'LOSO-CV Subject-Level ROC Curves — All {notebook_type} Models', fontsize=14, fontweight='bold', pad=12)
    plt.legend(loc='lower right', fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(output_dir, f'all_models_roc_curves_{notebook_type}.png')
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {path}")

    # --- 3. Train/Val Accuracy Curves (mean ± SD across folds) ---
    # Skip for ML models (no training histories)
    dl_results = [r for r in results_registry if not r.get('is_ml', False) and r['histories']]
    if dl_results:
        n_dl = len(dl_results)
        n_cols = min(4, n_dl)
        n_rows = int(np.ceil(n_dl / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), dpi=150)
        if n_dl == 1:
            axes = np.array([[axes]])
        elif n_rows == 1:
            axes = axes.reshape(1, -1)

        for i, result in enumerate(dl_results):
            name = result['model_name']
            histories = result['histories']
            row, col = i // n_cols, i % n_cols
            ax = axes[row, col] if n_rows > 1 else axes[0, col]

            train_accs = []
            val_accs = []
            for fold_key, hist in histories.items():
                train_accs.append(hist['accuracy'])
                val_accs.append(hist['val_accuracy'])

            max_len = max(len(a) for a in train_accs)
            train_arr = np.full((len(train_accs), max_len), np.nan)
            val_arr = np.full((len(val_accs), max_len), np.nan)
            for j, (ta, va) in enumerate(zip(train_accs, val_accs)):
                train_arr[j, :len(ta)] = ta
                val_arr[j, :len(va)] = va

            train_mean = np.nanmean(train_arr, axis=0)
            train_std = np.nanstd(train_arr, axis=0)
            val_mean = np.nanmean(val_arr, axis=0)
            val_std = np.nanstd(val_arr, axis=0)
            epochs = np.arange(1, max_len + 1)

            ax.plot(epochs, train_mean, color='#0279EE', lw=1.5, label='Train')
            ax.fill_between(epochs, train_mean - train_std, train_mean + train_std, color='#0279EE', alpha=0.15)
            ax.plot(epochs, val_mean, color='#FF9400', lw=1.5, label='Val')
            ax.fill_between(epochs, val_mean - val_std, val_mean + val_std, color='#FF9400', alpha=0.15)
            ax.set_title(name, fontsize=10, fontweight='bold')
            ax.set_xlabel('Epoch', fontsize=9)
            ax.set_ylabel('Accuracy', fontsize=9)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)

        for i in range(n_dl, n_rows * n_cols):
            row, col = i // n_cols, i % n_cols
            axes[row, col].axis('off') if n_rows > 1 else axes[0, col].axis('off')

        plt.suptitle(f'LOSO-CV Training & Validation Accuracy (Mean ± SD across folds) — {notebook_type} Models',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        path = os.path.join(output_dir, f'all_models_train_val_accuracy_{notebook_type}.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")
    else:
        print(f"  Skipped train/val accuracy plot (no DL models with histories)")

    # --- 4. Train Loss Curves (mean ± SD across folds) ---
    if dl_results:
        n_dl = len(dl_results)
        n_cols = min(4, n_dl)
        n_rows = int(np.ceil(n_dl / n_cols))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows), dpi=150)
        if n_dl == 1:
            axes = np.array([[axes]])
        elif n_rows == 1:
            axes = axes.reshape(1, -1)

        for i, result in enumerate(dl_results):
            name = result['model_name']
            histories = result['histories']
            row, col = i // n_cols, i % n_cols
            ax = axes[row, col] if n_rows > 1 else axes[0, col]

            train_losses = [hist['loss'] for hist in histories.values()]
            val_losses = [hist['val_loss'] for hist in histories.values()]

            max_len = max(len(l) for l in train_losses)
            train_arr = np.full((len(train_losses), max_len), np.nan)
            val_arr = np.full((len(val_losses), max_len), np.nan)
            for j, (tl, vl) in enumerate(zip(train_losses, val_losses)):
                train_arr[j, :len(tl)] = tl
                val_arr[j, :len(vl)] = vl

            train_mean = np.nanmean(train_arr, axis=0)
            train_std = np.nanstd(train_arr, axis=0)
            val_mean = np.nanmean(val_arr, axis=0)
            val_std = np.nanstd(val_arr, axis=0)
            epochs = np.arange(1, max_len + 1)

            ax.plot(epochs, train_mean, color='#0279EE', lw=1.5, label='Train')
            ax.fill_between(epochs, train_mean - train_std, train_mean + train_std, color='#0279EE', alpha=0.15)
            ax.plot(epochs, val_mean, color='#FF9400', lw=1.5, label='Val')
            ax.fill_between(epochs, val_mean - val_std, val_mean + val_std, color='#FF9400', alpha=0.15)
            ax.set_title(name, fontsize=10, fontweight='bold')
            ax.set_xlabel('Epoch', fontsize=9)
            ax.set_ylabel('Loss', fontsize=9)
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)

        for i in range(n_dl, n_rows * n_cols):
            row, col = i // n_cols, i % n_cols
            axes[row, col].axis('off') if n_rows > 1 else axes[0, col].axis('off')

        plt.suptitle(f'LOSO-CV Training & Validation Loss (Mean ± SD across folds) — {notebook_type} Models',
                     fontsize=13, fontweight='bold', y=1.01)
        plt.tight_layout()
        path = os.path.join(output_dir, f'all_models_train_loss_{notebook_type}.png')
        plt.savefig(path, dpi=200, bbox_inches='tight')
        plt.close()
        print(f"  Saved: {path}")
    else:
        print(f"  Skipped train loss plot (no DL models with histories)")


# ============================================================================
# 7. SUMMARY TABLES
# ============================================================================

def generate_summary_table(results_registry, output_dir, notebook_type='DL'):
    """Generate CSV summary files: cross-model comparison, per-subject accuracy, per-fold metrics."""
    import pandas as pd

    # --- Summary CSV (one row per model) ---
    summary_rows = []
    for result in results_registry:
        m = result['metrics']
        row = {
            'Model': m['model_name'],
            'Subject_Accuracy': m.get('subject_accuracy'),
            'Subject_Sensitivity_MDD': m.get('subject_sensitivity_mdd'),
            'Subject_Specificity_HC': m.get('subject_specificity_hc'),
            'Subject_F1_Macro': m.get('subject_f1_macro'),
            'Subject_ROC_AUC': m.get('subject_roc_auc'),
            'Subject_CI_95_Wilson_Lower': m['subject_accuracy_95ci_wilson'][0] if m.get('subject_accuracy_95ci_wilson') else None,
            'Subject_CI_95_Wilson_Upper': m['subject_accuracy_95ci_wilson'][1] if m.get('subject_accuracy_95ci_wilson') else None,
            'OOF_Accuracy': m['oof_accuracy'],
            'Per_Fold_Acc_Mean': m['per_fold_accuracy_mean'],
            'Per_Fold_Acc_SD': m['per_fold_accuracy_sd'],
            'CI_95_Wilson_Lower': m['oof_accuracy_95ci_wilson'][0],
            'CI_95_Wilson_Upper': m['oof_accuracy_95ci_wilson'][1],
            'CI_95_Bootstrap_Lower': m['per_fold_accuracy_95ci_bootstrap'][0],
            'CI_95_Bootstrap_Upper': m['per_fold_accuracy_95ci_bootstrap'][1],
            'Sensitivity_MDD': m['sensitivity_mdd'],
            'Specificity_HC': m['specificity_hc'],
            'F1_MDD': m['f1_mdd'],
            'F1_HC': m['f1_hc'],
            'F1_Macro': m['f1_macro'],
            'ROC_AUC_OOF': m['roc_auc_oof'],
            'Mean_Epochs_Trained': m['mean_epochs_trained'],
        }
        summary_rows.append(row)
    df_summary = pd.DataFrame(summary_rows)
    path = os.path.join(output_dir, f'loso_cv_summary_{notebook_type}.csv')
    df_summary.to_csv(path, index=False)
    print(f"  Saved: {path}")

    # --- Per-Subject Accuracy CSV (subjects × models, window + subject level) ---
    all_subjects = sorted(set(r['subject'] for result in results_registry for r in result['per_fold_results']))
    acc_matrix = {}
    subj_acc_matrix = {}
    for result in results_registry:
        name = result['model_name']
        acc_matrix[name] = {}
        subj_acc_matrix[name] = {}
        for fr in result['per_fold_results']:
            acc_matrix[name][fr['subject']] = fr['accuracy']
            subj_acc_matrix[name][fr['subject']] = fr.get('subject_accuracy', np.nan)

    df_subject = pd.DataFrame(index=all_subjects)
    for name in acc_matrix:
        df_subject[f'{name}_window'] = df_subject.index.map(lambda s: acc_matrix[name].get(s, np.nan))
        df_subject[f'{name}_subject'] = df_subject.index.map(lambda s: subj_acc_matrix[name].get(s, np.nan))
    path = os.path.join(output_dir, f'loso_cv_per_subject_accuracy_{notebook_type}.csv')
    df_subject.to_csv(path)
    print(f"  Saved: {path}")

    # --- Per-Fold Metrics CSV (all folds × all models) ---
    fold_rows = []
    for result in results_registry:
        name = result['model_name']
        for fr in result['per_fold_results']:
            fold_rows.append({
                'Model': name,
                'Fold': fr['fold'],
                'Subject': fr['subject'],
                'Label': 'MDD' if fr['label'] == 1 else 'HC',
                'Window_Accuracy': fr['accuracy'],
                'Window_Sensitivity': fr['sensitivity'],
                'Window_Specificity': fr['specificity'],
                'Subject_Pred_Prob': fr.get('subject_pred_prob'),
                'Subject_Pred': fr.get('subject_pred'),
                'Subject_Accuracy': fr.get('subject_accuracy'),
                'N_Samples': fr['n_samples'],
                'N_Correct': fr['n_correct'],
                'Epochs_Trained': fr['epochs_trained'],
            })
    df_folds = pd.DataFrame(fold_rows)
    path = os.path.join(output_dir, f'loso_cv_per_fold_metrics_{notebook_type}.csv')
    df_folds.to_csv(path, index=False)
    print(f"  Saved: {path}")

    # --- Print summary table ---
    print(f"\n{'='*110}")
    print(f"LOSO-CV SUMMARY — {notebook_type} Models")
    print(f"{'='*110}")
    print(f"{'Model':<20} {'SubjAcc':>8} {'SubjSens':>8} {'SubjSpec':>8} {'SubjAUC':>8} {'WinAcc':>8} {'WinSens':>8} {'WinSpec':>8} {'WinAUC':>8}")
    print(f"{'-'*110}")
    for _, row in df_summary.iterrows():
        subj_acc = row['Subject_Accuracy']
        subj_sens = row['Subject_Sensitivity_MDD']
        subj_spec = row['Subject_Specificity_HC']
        subj_auc = row['Subject_ROC_AUC']
        print(f"{row['Model']:<20} "
              f"{subj_acc:>8.3f} {subj_sens:>8.3f} {subj_spec:>8.3f} {subj_auc:>8.3f} "
              f"{row['OOF_Accuracy']:>8.3f} {row['Sensitivity_MDD']:>8.3f} "
              f"{row['Specificity_HC']:>8.3f} {row['ROC_AUC_OOF']:>8.3f}")
    print(f"{'='*110}")

# **SETUP SAVE AND SPLITS**

In [8]:
# ================================================================================================
# CELL 1 — MODMA EEG Dataset: Setup + Load + Preprocess + Subject-Wise Split + Save
# ================================================================================================
#
# CRITICAL METHODOLOGICAL CORRECTIONS IMPLEMENTED (OMNI-REVISION):
# -----------------------------------------------------------------
# 1. SUBJECT-WISE SPLITTING MOVED PRIOR TO ALL STATISTICAL TRANSFORMATIONS: 
#    Splitting now occurs immediately after temporal windowing.
# 2. ISOLATED IMPUTATION: NaN means are calculated strictly on the training partition 
#    and applied to the test partition, preventing statistical bleed.
# 3. ISOLATED PCA: If enabled, PCA components are derived solely from the training matrix.
# 4. NO GLOBAL NORMALIZATION: Per-window z-score is applied per-fold in the
#    LOSO-CV framework (preprocess_fold), avoiding double normalization and
#    ensuring each window is independently normalized for cross-subject generalization.
# ================================================================================================

import os
import re
import json
import warnings
from collections import defaultdict
from typing import Tuple, Dict, List, Optional
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
# StandardScaler removed — per-window z-score normalization is applied in the
# LOSO-CV framework (preprocess_fold) to avoid double normalization.

# ================================================================================================
# CONFIGURATION 
# ================================================================================================
DATA_DIR = "/kaggle/input/datasets/mimino12/preprocessed-raw-mat-csv/mat-csv-actual/mat-csv-actual/csv_files_from_mat2"
OUTPUT_DIR = "/kaggle/working/mat-main-results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ================================================================================================
# HYPERPARAMETERS
# ================================================================================================
SAMPLE_FRAC = 1.0 
TEST_SIZE = 0.2 
SEED = 42 
VALIDATE_NO_LEAKAGE = True 

USE_IPCA = False 
IPCA_COMPONENTS = 128 
IPCA_BATCH = 5000

WINDOW_SIZE = 250 
STRIDE = 125 

np.random.seed(SEED)
tf.random.set_seed(SEED)

# ================================================================================================
# GPU CONFIGURATION
# ================================================================================================
gpus = tf.config.experimental.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
        print(f"✓ GPU memory growth enabled: {g}")
    except Exception as e:
        print(f"✗ GPU config failed: {e}")
if not gpus:
    print("ℹ No GPU detected — running on CPU")

# ================================================================================================
# HELPER FUNCTIONS
# ================================================================================================
def extract_subject_id_and_label(filename: str) -> Tuple[Optional[str], Optional[int]]:
    basename = os.path.splitext(filename)[0]
    match = re.match(r'^(020[13])[-_]?(\d+)', basename)
    if match:
        return f"{match.group(1)}_{match.group(2)}", 1 if match.group(1) == "0201" else 0
    
    match = re.match(r'^(MDD|HC)[-_]?(.+)', basename, re.IGNORECASE)
    if match:
        return f"{match.group(1).upper()}_{match.group(2)}", 1 if match.group(1).upper() == "MDD" else 0
    
    return basename, None

def detect_eeg_columns(columns: List[str]) -> List[str]:
    regex = re.compile(r'^(?:EEG[_\-\s]?|E[_\-\s]?)(0*)(\d{1,3})$', flags=re.IGNORECASE)
    found = {}
    for col in columns:
        match = regex.match(col.strip())
        if match:
            electrode_num = int(match.group(2))
            if 1 <= electrode_num <= 128:
                found[electrode_num] = col
    if found:
        return [found[i] for i in sorted(found.keys())]
    
    fallback = [c for c in columns if re.match(r'^(E|EEG)\d+', c, flags=re.IGNORECASE)]
    return fallback if fallback else []

def verify_no_subject_leakage(train_subjects: set, test_subjects: set) -> bool:
    overlap = train_subjects & test_subjects
    if overlap:
        raise AssertionError(f"DATA LEAKAGE DETECTED! Overlapping subjects: {overlap}.")
    return True

def subject_wise_train_test_split(
    X: np.ndarray, y: np.ndarray, subject_ids: np.ndarray, test_size: float = 0.2, random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    unique_subjects = np.unique(subject_ids)
    subject_labels = []
    
    for subj in unique_subjects:
        subj_mask = subject_ids == subj
        subj_label = y[subj_mask][0] 
        subject_labels.append(subj_label)
        
    subject_labels = np.array(subject_labels)
    
    subjects_train, subjects_test = train_test_split(
        unique_subjects, test_size=test_size, stratify=subject_labels, random_state=random_state
    )
    
    train_mask = np.isin(subject_ids, subjects_train)
    test_mask = np.isin(subject_ids, subjects_test)
    
    return X[train_mask], X[test_mask], y[train_mask], y[test_mask], subject_ids[train_mask], subject_ids[test_mask]

# ================================================================================================
# SPECTRAL FEATURE EXTRACTION (v3)
# ================================================================================================
from scipy.signal import welch as _welch

BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta': (13, 30),
    'gamma': (30, 45),
}
BAND_NAMES = list(BANDS.keys())
SPECTRAL_FEATURE_NAMES = (
    [f'{b}_abs' for b in BAND_NAMES] +
    [f'{b}_rel' for b in BAND_NAMES] +
    ['theta_beta_ratio', 'alpha_beta_ratio', 'theta_alpha_ratio']
)

def compute_spectral_features(X, fs=250, batch_size=500):
    """
    Compute Welch's PSD spectral features per window per channel on RAW data.
    Returns (n_samples, n_channels, 13) array: 5 abs powers (log10) +
    5 relative powers + 3 band ratios.
    """
    n_samples, n_timesteps, n_channels = X.shape
    nperseg = min(n_timesteps, 250)
    noverlap = nperseg // 2
    n_features = len(BANDS) * 2 + 3
    features = np.zeros((n_samples, n_channels, n_features), dtype=np.float32)

    for start in range(0, n_samples, batch_size):
        end = min(start + batch_size, n_samples)
        X_batch = X[start:end].astype(np.float64)
        freqs, psd = _welch(X_batch, fs=fs, nperseg=nperseg, noverlap=noverlap,
                            window='hann', axis=1, detrend='constant')
        band_powers = {}
        for band_name, (fmin, fmax) in BANDS.items():
            mask = (freqs >= fmin) & (freqs < fmax)
            band_powers[band_name] = np.sum(psd[:, mask, :], axis=1)
        total_mask = (freqs >= 1) & (freqs < 45)
        total_power = np.sum(psd[:, total_mask, :], axis=1)

        feat_idx = 0
        for band_name in BAND_NAMES:
            features[start:end, :, feat_idx] = np.log10(band_powers[band_name] + 1e-10)
            feat_idx += 1
        for band_name in BAND_NAMES:
            features[start:end, :, feat_idx] = band_powers[band_name] / (total_power + 1e-10)
            feat_idx += 1
        features[start:end, :, feat_idx] = band_powers['theta'] / (band_powers['beta'] + 1e-10); feat_idx += 1
        features[start:end, :, feat_idx] = band_powers['alpha'] / (band_powers['beta'] + 1e-10); feat_idx += 1
        features[start:end, :, feat_idx] = band_powers['theta'] / (band_powers['alpha'] + 1e-10); feat_idx += 1

    return np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)

# ================================================================================================
# STEP 1: LOAD CSV FILES AND EXTRACT SUBJECT INFORMATION
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 1: Loading CSV files\n" + "=" * 80)
csv_files = sorted([f for f in os.listdir(DATA_DIR) if f.lower().endswith('.csv')])
if not csv_files: raise RuntimeError(f"No CSV files found in: {DATA_DIR}")

subject_data, subject_labels_dict = defaultdict(list), {}

for filename in csv_files:
    filepath = os.path.join(DATA_DIR, filename)
    subject_id, final_label = extract_subject_id_and_label(filename)
    
    if subject_id is None or final_label is None: continue
    subject_labels_dict[subject_id] = final_label
    
    df = pd.read_csv(filepath, engine='python')
    if SAMPLE_FRAC < 1.0: df = df.sample(frac=SAMPLE_FRAC, random_state=SEED)
    subject_data[subject_id].append(df)

all_dfs = []
for subject_id, df_list in subject_data.items():
    combined_df = pd.concat(df_list, ignore_index=True)
    all_dfs.append(combined_df)
combined = pd.concat(all_dfs, ignore_index=True)

print(f"✓ Loaded {len(subject_labels_dict)} subjects. Total samples: {len(combined):,}")

# ================================================================================================
# STEP 2: DETECT AND VALIDATE EEG COLUMNS
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 2: Detecting EEG electrode columns\n" + "=" * 80)
eeg_cols = detect_eeg_columns(combined.columns)
metadata_cols = {'time', 'condition', 'label', 'epoch', 'target', 'cond'}
feature_cols = [c for c in eeg_cols if c.lower() not in {m.lower() for m in metadata_cols}]
print(f"✓ Using {len(feature_cols)} channels as features.")

# ================================================================================================
# STEP 3: TEMPORAL WINDOWING (YIELDS UNSPLIT TENSORS)
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 3: Temporal Windowing\n" + "=" * 80)
all_windows, all_y, all_subject_ids = [], [], []

for subject_id, df_list in subject_data.items():
    X_subject = pd.concat(df_list, ignore_index=True)[feature_cols].to_numpy(dtype=np.float32)
    num_windows = max(0, (X_subject.shape[0] - WINDOW_SIZE) // STRIDE + 1)
    if num_windows > 0:
        windows = np.array([X_subject[i:i + WINDOW_SIZE] for i in range(0, X_subject.shape[0] - WINDOW_SIZE + 1, STRIDE)])
        all_windows.append(windows)
        all_y.extend([subject_labels_dict[subject_id]] * num_windows)
        all_subject_ids.extend([subject_id] * num_windows)

X_unsplit = np.concatenate(all_windows, axis=0)
y = np.array(all_y, dtype=np.int32)
subject_ids = np.array(all_subject_ids)
print(f"✓ Windowed shape: {X_unsplit.shape}")

# ================================================================================================
# STEP 4: SUBJECT-WISE TRAIN-TEST SPLIT & VALIDATION (MOVED UP)
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 4: Subject-wise train-test split & Leakage Validation\n" + "=" * 80)
X_train, X_test, y_train, y_test, subj_train, subj_test = subject_wise_train_test_split(
    X_unsplit, y, subject_ids, test_size=TEST_SIZE, random_state=SEED
)

train_subject_set, test_subject_set = set(np.unique(subj_train)), set(np.unique(subj_test))
if VALIDATE_NO_LEAKAGE: verify_no_subject_leakage(train_subject_set, test_subject_set)
print("✓ VALIDATION PASSED: Zero subject overlap between sets.")

# ================================================================================================
# STEP 5: HANDLE MISSING VALUES (NO STATISTICAL LEAKAGE)
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 5: Handling missing values via Training-Set Means\n" + "=" * 80)

# Calculate means strictly on X_train
X_train_reshaped = X_train.reshape(-1, X_train.shape[2])
col_means = np.nanmean(X_train_reshaped, axis=0)

# Apply to Train
nan_indices_train = np.where(np.isnan(X_train_reshaped))
X_train_reshaped[nan_indices_train] = np.take(col_means, nan_indices_train[1])
X_train = X_train_reshaped.reshape(X_train.shape)

# Apply to Test
X_test_reshaped = X_test.reshape(-1, X_test.shape[2])
nan_indices_test = np.where(np.isnan(X_test_reshaped))
X_test_reshaped[nan_indices_test] = np.take(col_means, nan_indices_test[1])
X_test = X_test_reshaped.reshape(X_test.shape)

print(f"✓ Imputed NaN values using training set distribution only.")

# ================================================================================================
# STEP 6: OPTIONAL PCA 
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 6: Dimensionality reduction (PCA)\n" + "=" * 80)
if USE_IPCA and IPCA_COMPONENTS is not None:
    from sklearn.decomposition import IncrementalPCA
    ipca = IncrementalPCA(n_components=IPCA_COMPONENTS)
    
    # Fit on Train
    X_train_flat = X_train.reshape(-1, X_train.shape[2])
    for i in range(0, X_train_flat.shape[0], IPCA_BATCH):
        ipca.partial_fit(X_train_flat[i:i + IPCA_BATCH])
        
    # Transform Train
    X_train_red = np.empty((X_train_flat.shape[0], IPCA_COMPONENTS), dtype=np.float32)
    for i in range(0, X_train_flat.shape[0], IPCA_BATCH):
        X_train_red[i:i + IPCA_BATCH] = ipca.transform(X_train_flat[i:i + IPCA_BATCH])
    X_train = X_train_red.reshape(X_train.shape[0], X_train.shape[1], IPCA_COMPONENTS)

    # Transform Test
    X_test_flat = X_test.reshape(-1, X_test.shape[2])
    X_test_red = np.empty((X_test_flat.shape[0], IPCA_COMPONENTS), dtype=np.float32)
    for i in range(0, X_test_flat.shape[0], IPCA_BATCH):
        X_test_red[i:i + IPCA_BATCH] = ipca.transform(X_test_flat[i:i + IPCA_BATCH])
    X_test = X_test_red.reshape(X_test.shape[0], X_test.shape[1], IPCA_COMPONENTS)
    
    print(f"✓ PCA complete. Train shape: {X_train.shape}, Test shape: {X_test.shape}")
else:
    print("✓ PCA SKIPPED — preserving raw EEG channel structure for CNN/TCN.")

# ================================================================================================
# STEP 7: NORMALIZATION — REMOVED (handled per-fold in LOSO-CV framework)
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 7: Normalization — skipped (per-window z-score in framework)\n" + "=" * 80)
print("✓ No global StandardScaler — per-window z-score applied in preprocess_fold() per fold.")
# ================================================================================================
# STEP 7.5: SPECTRAL FEATURE EXTRACTION (v3 — computed on RAW, non-z-scored data)
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 7.5: Spectral feature extraction (Welch PSD)\n" + "=" * 80)
print("  Computing spectral features for training set...")
spectral_train = compute_spectral_features(X_train, fs=250, batch_size=500)
print(f"  Train spectral shape: {spectral_train.shape}")
print("  Computing spectral features for test set...")
spectral_test = compute_spectral_features(X_test, fs=250, batch_size=500)
print(f"  Test spectral shape: {spectral_test.shape}")
print(f"  Features per channel: {spectral_train.shape[2]} (5 abs + 5 rel + 3 ratios)")
print(f"  Feature range: [{spectral_train.min():.4f}, {spectral_train.max():.4f}]")
print("✓ Spectral features computed on raw (non-z-scored) data — preserves amplitude info.")



# ================================================================================================
# STEP 8: SAVE PROCESSED DATA
# ================================================================================================
print("\n" + "=" * 80 + "\nSTEP 8: Saving processed data\n" + "=" * 80)
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'mat_data_split.npz'),
    X_train=X_train, X_test=X_test, y_train=y_train, y_test=y_test,
    subjects_train=subj_train, subjects_test=subj_test,
    feature_columns=np.array(feature_cols),
    spectral_train=spectral_train, spectral_test=spectral_test,
    spectral_feature_names=np.array(SPECTRAL_FEATURE_NAMES)
)

metadata = {
    "dataset": "MODMA",
    "preprocessing": {"window_size": WINDOW_SIZE, "stride": STRIDE, "pca_applied": USE_IPCA, "spectral_features": True, "n_spectral_features": spectral_train.shape[2]},
    "subjects": {"train_subjects": sorted(list(train_subject_set)), "test_subjects": sorted(list(test_subject_set))},
    "shapes": {"X_train": list(X_train.shape), "X_test": list(X_test.shape)}
}
with open(os.path.join(OUTPUT_DIR, 'preprocessing_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"✅ PREPROCESSING COMPLETE. ZERO LEAKAGE ARCHITECTURE SECURED.")

✓ GPU memory growth enabled: PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')
✓ GPU memory growth enabled: PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')

STEP 1: Loading CSV files
✓ Loaded 35 subjects. Total samples: 2,643,030

STEP 2: Detecting EEG electrode columns
✓ Using 128 channels as features.

STEP 3: Temporal Windowing
✓ Windowed shape: (21095, 250, 128)

STEP 4: Subject-wise train-test split & Leakage Validation
✓ VALIDATION PASSED: Zero subject overlap between sets.

STEP 5: Handling missing values via Training-Set Means
✓ Imputed NaN values using training set distribution only.

STEP 6: Dimensionality reduction (PCA)
✓ PCA SKIPPED — preserving raw EEG channel structure for CNN/TCN.

STEP 7: Normalization — skipped (per-window z-score in framework)
✓ No global StandardScaler — per-window z-score applied in preprocess_fold() per fold.

STEP 7.5: Spectral feature extraction (Welch PSD)
  Computing spectral features for training set...
  Train

# **STATISTICAL ANALYSIS**

Graph-theoretic analysis: Phase Lag Index (PLI) connectivity, betweenness centrality, clustering coefficient, and FDR-corrected nodal t-tests. Preserved from the original DL notebook.

In [10]:
# ================================================================================================
# MODMA EEG GRAPH-THEORETIC ANALYSIS — RESTING-STATE PIPELINE
# Addressing Reviewer 1 (Connectivity Metrics) & Reviewer 3 (Post-Hoc Localized Statistics)
# ================================================================================================
import os
import gc
import numpy as np
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import hilbert
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import fdrcorrection
from tqdm import tqdm

# ================================================================================================
# CONFIGURATION
# ================================================================================================
OUTPUT_DIR = "/kaggle/working/mat-main-results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# Ensure your resting-state file is targeted here
DATA_PATH = os.path.join(OUTPUT_DIR, 'mat_data_split.npz') 
TOP_EDGE_PERCENTILE = 80  # Top 20% strongest connections (Adjacency Thresholding - Rev 1)

# ================================================================================================
# CORE COMPUTATION FUNCTIONS
# ================================================================================================
def compute_subject_mean_pli(windows: np.ndarray, batch_size: int = 50) -> np.ndarray:
    """
    Computes Phase Lag Index (PLI) to establish functional connectivity.
    Addressing Rev 1: Explicitly defining the connectivity metric.
    """
    n_windows, n_channels, _ = windows.shape
    sum_pli = np.zeros((n_channels, n_channels), dtype=np.float32)
    
    for i in range(0, n_windows, batch_size):
        batch = windows[i : i + batch_size]
        analytic = hilbert(batch, axis=-1)
        cross_spectrum = analytic[:, :, None, :] * np.conj(analytic[:, None, :, :])
        pli_batch = np.abs(np.mean(np.sign(np.imag(cross_spectrum)), axis=-1))
        sum_pli += np.sum(pli_batch, axis=0)
        del batch, analytic, cross_spectrum, pli_batch
    
    mean_pli = sum_pli / n_windows
    np.fill_diagonal(mean_pli, 0)
    return mean_pli

def extract_node_metrics(adj_matrix: np.ndarray) -> tuple:
    """
    Extracts node-wise Betweenness Centrality and Clustering Coefficient.
    Addressing Rev 1: Explicit graph statistics.
    """
    G = nx.from_numpy_array(adj_matrix)
    
    bc_dict = nx.betweenness_centrality(G, normalized=True)
    bc_array = np.array([bc_dict[node] for node in range(len(G.nodes))])
    
    cc_dict = nx.clustering(G)
    cc_array = np.array([cc_dict[node] for node in range(len(G.nodes))])
    
    return bc_array, cc_array

# ================================================================================================
# TABLE-TO-PNG RENDERER
# ================================================================================================
def export_dataframe_to_png(df: pd.DataFrame, filename: str, title: str, top_n: int = 25):
    """
    Renders top N rows of localized post-hoc stats as a PNG.
    Addressing Rev 3: Typo fix in headers and deep statistical reporting.
    """
    df_subset = df.head(top_n).copy()
    
    df_subset['MDD_Mean'] = df_subset['MDD_Mean'].map('{:.4f}'.format)
    df_subset['HC_Mean'] = df_subset['HC_Mean'].map('{:.4f}'.format)
    df_subset['t_statistic'] = df_subset['t_statistic'].map('{:.3f}'.format)
    df_subset['p_uncorrected'] = df_subset['p_uncorrected'].map('{:.4f}'.format)
    df_subset['p_FDR_corrected'] = df_subset['p_FDR_corrected'].map('{:.4f}'.format)
    
    fig, ax = plt.subplots(figsize=(14, 0.4 * len(df_subset) + 1.5))
    ax.axis('tight')
    ax.axis('off')
    ax.set_title(f"{title} (Top {top_n} Nodes)", fontsize=16, weight='bold', pad=20)
    
    table = ax.table(
        cellText=df_subset.values,
        colLabels=df_subset.columns, # Typo "coeffient" is avoided by programmatic generation
        loc='center',
        cellLoc='center'
    )
    
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1.2, 1.5)
    
    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_text_props(weight='bold', color='white')
            cell.set_facecolor('#4c72b0')
        else:
            cell.set_facecolor('#f2f2f2' if row % 2 == 0 else 'white')
            
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300, bbox_inches='tight')
    plt.close()

# ================================================================================================
# PHASE 1: DATA LOADING & ADJACENCY CONSTRUCTION (Rev 1)
# ================================================================================================
print("=" * 80)
print("RESTING-STATE NETWORK TOPOLOGY EXTRACTION")
print("=" * 80)

data = np.load(DATA_PATH, allow_pickle=True)

X_full = np.concatenate((data['X_train'], data['X_test']), axis=0)
y_full = np.concatenate((data['y_train'], data['y_test']), axis=0)
subjects_full = np.concatenate((data['subjects_train'], data['subjects_test']), axis=0)

channel_names = list(data['feature_columns']) if 'feature_columns' in data else [f"E{i+1}" for i in range(X_full.shape[2])]

if X_full.ndim == 4 and X_full.shape[1] == 1:
    X_full = np.squeeze(X_full, axis=1)

unique_subjects = np.unique(subjects_full)
subject_metrics = []

for subj_id in tqdm(unique_subjects, desc="Constructing Adjacency Matrices"):
    mask = (subjects_full == subj_id)
    subj_windows = X_full[mask]
    subj_label = y_full[mask][0]
    
    mean_pli = compute_subject_mean_pli(np.transpose(subj_windows, (0, 2, 1)))
    upper_tri = mean_pli[np.triu_indices_from(mean_pli, k=1)]
    threshold = np.percentile(upper_tri, TOP_EDGE_PERCENTILE)
    adj_matrix = (mean_pli > threshold).astype(float)
    
    bc_array, cc_array = extract_node_metrics(adj_matrix)
    
    subject_metrics.append({
        'subject_id': subj_id,
        'label': subj_label,
        'betweenness': bc_array,
        'clustering': cc_array
    })
    
    del subj_windows, mean_pli, adj_matrix
    gc.collect()

labels = np.array([m['label'] for m in subject_metrics])
bc_matrix = np.vstack([m['betweenness'] for m in subject_metrics])
cc_matrix = np.vstack([m['clustering'] for m in subject_metrics])

mdd_mask = (labels == 1)
hc_mask = (labels == 0)

# ================================================================================================
# PHASE 2: GLOBAL STATISTICS & POST-HOC NODAL ANALYSIS (Rev 3)
# ================================================================================================
print("\n" + "=" * 80)
print("STATISTICAL VALIDATION: GLOBAL VS LOCAL (Rev 3)")
print("=" * 80)

# 1. Global (Average) Statistics to mirror manuscript claims
global_bc_mdd = np.mean(bc_matrix[mdd_mask], axis=1)
global_bc_hc = np.mean(bc_matrix[hc_mask], axis=1)
t_glob_bc, p_glob_bc = ttest_ind(global_bc_mdd, global_bc_hc, equal_var=False)

global_cc_mdd = np.mean(cc_matrix[mdd_mask], axis=1)
global_cc_hc = np.mean(cc_matrix[hc_mask], axis=1)
t_glob_cc, p_glob_cc = ttest_ind(global_cc_mdd, global_cc_hc, equal_var=False)

print(f"Global Betweenness: t={t_glob_bc:.2f}, p={p_glob_bc:.2f}")
print(f"Global Clustering:  t={t_glob_cc:.2f}, p={p_glob_cc:.2f}")
print("Note: Non-significant global metrics do not preclude localized disruptions.\n")

# 2. Targeted Node-Wise Post-Hoc Analysis
def compute_and_export_node_stats(mdd_data, hc_data, metric_name, filename, ch_names):
    n_nodes = mdd_data.shape[1]
    p_values, t_stats = np.zeros(n_nodes), np.zeros(n_nodes)
    mdd_means, hc_means = np.mean(mdd_data, axis=0), np.mean(hc_data, axis=0)
    
    for i in range(n_nodes):
        t, p = ttest_ind(mdd_data[:, i], hc_data[:, i], equal_var=False)
        t_stats[i], p_values[i] = t, p
        
    rejected, p_fdr = fdrcorrection(p_values, alpha=0.05, method='indep')
    
    print(f"{metric_name} Nodal Disruptions:")
    print(f" - Uncorrected (p < 0.05): {np.sum(p_values < 0.05)}")
    print(f" - FDR Corrected (p < 0.05): {np.sum(rejected)}")
    
    results = [{
        'Electrode': ch_names[i],
        'Metric': metric_name,
        'MDD_Mean': mdd_means[i],
        'HC_Mean': hc_means[i],
        't_statistic': t_stats[i],
        'p_uncorrected': p_values[i],
        'p_FDR_corrected': p_fdr[i],
        'Survives_FDR': rejected[i],
        'Direction': "MDD > HC" if t_stats[i] > 0 else "MDD < HC"
    } for i in range(n_nodes)]
            
    df_results = pd.DataFrame(results).sort_values(by='p_uncorrected')
    export_dataframe_to_png(df_results, filename, f"Resting-State Post-Hoc: {metric_name.replace('_', ' ')}")
    return df_results

df_bc_rest = compute_and_export_node_stats(
    bc_matrix[mdd_mask], bc_matrix[hc_mask], 
    "Betweenness_Centrality", "table_resting_post_hoc_bc.png", channel_names
)
df_cc_rest = compute_and_export_node_stats(
    cc_matrix[mdd_mask], cc_matrix[hc_mask], 
    "Clustering_Coefficient", "table_resting_post_hoc_cc.png", channel_names
)

RESTING-STATE NETWORK TOPOLOGY EXTRACTION


Constructing Adjacency Matrices: 100%|██████████| 35/35 [07:58<00:00, 13.68s/it]
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.



STATISTICAL VALIDATION: GLOBAL VS LOCAL (Rev 3)
Global Betweenness: t=1.00, p=0.32
Global Clustering:  t=0.41, p=0.69
Note: Non-significant global metrics do not preclude localized disruptions.

Betweenness_Centrality Nodal Disruptions:
 - Uncorrected (p < 0.05): 1
 - FDR Corrected (p < 0.05): 0


findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font f

Clustering_Coefficient Nodal Disruptions:
 - Uncorrected (p < 0.05): 4
 - FDR Corrected (p < 0.05): 0


findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font f

In [11]:
# ================================================================================================
# VISUALIZATION FUNCTIONS (DISTRIBUTION & RANK-ORDER)
# ================================================================================================

def plot_subject_level_global_metrics(mdd_global, hc_global, metric_name, filename, ylabel):
    """
    Generates a boxplot with overlaid subject data points to visualize variance.
    Matches: plot_subject_level_betweenness_main.png
    """
    # Construct DataFrame for Seaborn
    df_vis = pd.DataFrame({
        metric_name: np.concatenate([mdd_global, hc_global]),
        'Diagnosis': ['MDD'] * len(mdd_global) + ['HC'] * len(hc_global)
    })
    
    plt.figure(figsize=(8, 6), dpi=300)
    
    # Create Boxplot
    sns.boxplot(
        x='Diagnosis', y=metric_name, data=df_vis,
        palette=['#f59b9b', '#9bcaee'], # Matching your specific color hexes
        boxprops=dict(alpha=0.9),
        showfliers=False, 
        width=0.7
    )
    
    # Overlay individual subject points
    sns.stripplot(
        x='Diagnosis', y=metric_name, data=df_vis,
        color='black', alpha=0.5, size=6, jitter=True
    )
    
    plt.title(f'Subject-Level Global {metric_name.replace("_", " ")} (MDD vs HC)', fontsize=14, pad=15)
    plt.ylabel(ylabel, fontsize=12)
    plt.xlabel('Diagnosis', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), bbox_inches='tight')
    plt.close()

def plot_top_nodes_tstats(df_stats, metric_name, filename, top_n=10):
    """
    Generates a barplot of the highest driving nodes based on absolute t-statistic.
    Matches: plot_top_10_nodes_bc_main.png
    """
    # Sort by absolute t-statistic to find the strongest drivers, then take top N
    df_sorted = df_stats.copy()
    df_sorted['abs_t'] = df_sorted['t_statistic'].abs()
    df_top = df_sorted.sort_values(by='abs_t', ascending=False).head(top_n)
    
    plt.figure(figsize=(10, 6), dpi=300)
    
    # Create Barplot
    sns.barplot(
        x='Electrode', y='t_statistic', hue='Direction',
        data=df_top, dodge=False, palette='tab10' # Adjust palette as needed
    )
    
    plt.title(f'Top {top_n} Nodes Driving {metric_name.replace("_", " ")} Differences (Uncorrected)', fontsize=14, pad=15)
    plt.ylabel('t_statistic', fontsize=12)
    plt.xlabel('Electrode', fontsize=12)
    plt.xticks(rotation=45)
    plt.legend(title='Direction', loc='upper right')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), bbox_inches='tight')
    plt.close()

# ================================================================================================
# EXECUTION BLOCK (Append to the end of Phase 2)
# ================================================================================================

print("Generating distributional and nodal rank visualizations...")

# 1. Subject-Level Boxplot Execution
plot_subject_level_global_metrics(
    global_bc_mdd, global_bc_hc, 
    "Betweenness", "plot_subject_level_betweenness_main.png", 
    "Mean Subject Betweenness"
)

# 2. Top 10 Nodes Barplot Execution
# (Relies on df_bc_rest generated by your compute_and_export_node_stats function)
plot_top_nodes_tstats(
    df_bc_rest, "Betweenness_Centrality", "plot_top_10_nodes_bc_main.png", top_n=10
)

print("Visualizations successfully rendered to OUTPUT_DIR.")

findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.


Generating distributional and nodal rank visualizations...


findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font family 'Arimo' not found.
findfont: Font f

Visualizations successfully rendered to OUTPUT_DIR.


# **LOSO-CV DATA PREPARATION**

Load the preprocessed npz and reconstruct the full dataset with subject groups for Leave-One-Subject-Out cross-validation.

In [12]:
# ================================================================================================
# LOSO-CV DATA PREPARATION
# ================================================================================================
NPZ_PATH = "/kaggle/working/mat-main-results/mat_data_split.npz"
LOSO_BASE = "/kaggle/working/mat-main-results/loso_cv"
LOSO_DIR = os.path.join(LOSO_BASE, "DL_models")
os.makedirs(LOSO_DIR, exist_ok=True)

X, y, groups, feature_cols = prepare_loso_data(NPZ_PATH)
X_spec, y_spec, groups_spec, spec_names = prepare_spectral_loso_data(NPZ_PATH)
results_registry = []

  Loaded 21095 samples, 35 subjects, 128 channels
  MDD subjects: 22, HC subjects: 13
  Loaded spectral: 21095 samples, 35 subjects, 128 channels, 13 features


# **LOSO-CV: TCN**

In [ ]:
# ================================================================================================
# LOSO-CV: TCN (Temporal Convolutional Network) — v2 regularized
# ================================================================================================
from tensorflow.keras.layers import (Dense, Dropout, BatchNormalization, Activation,
                                       SpatialDropout1D, Input, Conv1D, GlobalAveragePooling1D, GaussianNoise)
from tensorflow.keras.models import Model

def TCN_block(x, filters, kernel_size, dilation_rate):
    x = Conv1D(filters, kernel_size, padding='causal', dilation_rate=dilation_rate,
               kernel_regularizer=L2_REG)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SpatialDropout1D(SPATIAL_DROPOUT)(x)
    return x

def build_tcn(input_shape):
    inputs = Input(shape=input_shape)
    x = GaussianNoise(GAUSSIAN_NOISE_STD)(inputs)
    x = TCN_block(x, filters=32, kernel_size=3, dilation_rate=1)
    x = TCN_block(x, filters=32, kernel_size=3, dilation_rate=2)
    x = TCN_block(x, filters=32, kernel_size=3, dilation_rate=4)
    x = TCN_block(x, filters=32, kernel_size=3, dilation_rate=8)
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation='relu', kernel_regularizer=L2_REG)(x)
    x = Dropout(DROPOUT_RATE)(x)
    outputs = Dense(1, activation='sigmoid')(x)
    model = Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
                  loss=tf.keras.losses.BinaryCrossentropy(label_smoothing=LABEL_SMOOTHING),
                  metrics=['accuracy'])
    return model

# v3: Multi-modal (raw EEG + spectral features via 19 standard 10-20 channels)
build_tcn_mm = wrap_multimodal(build_tcn, spectral_dim=247)

result_tcn = run_loso_cv('TCN', build_tcn_mm, X, y, groups,
                          input_mode='3d', max_epochs=80, batch_size=32,
                          output_dir=LOSO_DIR, special=None,
                          results_registry=results_registry,
                          X_spectral=X_spec, use_bootstrap=True)


LOSO-CV: TCN | 35 folds | input=3d | special=None

--- Fold 1/35 | Subject: 0201_0002 | Label: MDD | Samples: 599 ---
  Val subjects: 0201_0004 (MDD), 0203_0002 (HC) | Inner train: 19292 | Val: 1204


I0000 00:00:1787585351.095043      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1787585351.105651      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/80
 22/603 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.4610 - loss: 0.9646 

I0000 00:00:1787585371.777868     174 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


598/603 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.6172 - loss: 0.7504
Epoch 1: val_loss improved from None to 0.92768, saving model to /kaggle/working/mat-main-results/loso_cv/DL_models/TCN/fold_1_subject_0201_0002_best.weights.h5

Epoch 1: finished saving model to /kaggle/working/mat-main-results/loso_cv/DL_models/TCN/fold_1_subject_0201_0002_best.weights.h5
603/603 ━━━━━━━━━━━━━━━━━━━━ 34s 27ms/step - accuracy: 0.6943 - loss: 0.6574 - val_accuracy: 0.4875 - val_loss: 0.9277 - learning_rate: 5.0000e-04
Epoch 2/80
598/603 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8024 - loss: 0.5301
Epoch 2: val_loss did not improve from 0.92768
603/603 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8313 - loss: 0.5002 - val_accuracy: 0.5033 - val_loss: 1.0838 - learning_rate: 5.0000e-04
Epoch 3/80
601/603 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8841 - loss: 0.4397
Epoch 3: val_loss did not improve from 0.92768
603/603 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.8929 - loss: 0.4263